In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
df = pd.read_csv("../data/MLTempDataset1.csv", index_col=0)
df["Datetime"] = pd.to_datetime(df["Datetime"])
df = df.sort_values("Datetime").reset_index(drop=True)

In [3]:
assert (df["Datetime"].diff().dropna() == pd.Timedelta("1h")).all(), "Zaman serisinde boşluk var"
 
series = df["Hourly_Temp"].values.astype("float32").reshape(-1, 1)
print("Toplam gözlem:", len(series))
 

Toplam gözlem: 7056


In [4]:
n = len(series)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

In [5]:
train_raw = series[:train_end]
val_raw = series[train_end:val_end]
test_raw = series[val_end:]
print(f"Train: {len(train_raw)}  Val: {len(val_raw)}  Test: {len(test_raw)}")

Train: 4939  Val: 1058  Test: 1059


In [6]:
scaler = MinMaxScaler()
train_s = scaler.fit_transform(train_raw)
val_s = scaler.transform(val_raw)
test_s = scaler.transform(test_raw)

In [7]:
LOOKBACK = 24


In [8]:
def make_windows(data, lookback=LOOKBACK):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i])
        y.append(data[i])
    return np.array(X), np.array(y)

In [9]:
X_train, y_train = make_windows(train_s)
X_val, y_val = make_windows(np.vstack([train_s[-LOOKBACK:], val_s]))
X_test, y_test = make_windows(np.vstack([val_s[-LOOKBACK:], test_s]))

In [10]:
print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

X_train: (4915, 24, 1) X_val: (1058, 24, 1) X_test: (1059, 24, 1)


In [11]:
def evaluate(y_true_s, y_pred_s, name):
    y_true = scaler.inverse_transform(y_true_s.reshape(-1, 1)).ravel()
    y_pred = scaler.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:<22} MAE: {mae:.3f}   RMSE: {rmse:.3f}")
    return mae, rmse

In [12]:
baseline_pred = X_test[:, -1, 0]       
evaluate(y_test.ravel(), baseline_pred, "Naive baseline")

Naive baseline         MAE: 0.705   RMSE: 1.784


(0.7046988010406494, np.float64(1.7837188075331645))

In [13]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [14]:
tf.random.set_seed(42)
np.random.seed(42)

In [15]:
model = models.Sequential([
    layers.Input(shape=(LOOKBACK, 1)),
    layers.LSTM(64),
    layers.Dense(1),
])

In [16]:
model.compile(optimizer="adam", loss="mse", metrics=["mae"])


In [17]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 64)                  │          16,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 16,961 (66.25 KB)

 Trainable params: 16,961 (66.25 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
early = callbacks.EarlyStopping(monitor="val_loss", patience=8,
                                restore_best_weights=True, verbose=1)

In [19]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    shuffle=False,        
    callbacks=[early],
    verbose=2,
)

Epoch 1/100
77/77 - 11s - 143ms/step - loss: 0.0369 - mae: 0.1513 - val_loss: 0.0114 - val_mae: 0.0850
Epoch 2/100
77/77 - 4s - 56ms/step - loss: 0.0221 - mae: 0.1107 - val_loss: 0.0084 - val_mae: 0.0653
Epoch 3/100
77/77 - 5s - 64ms/step - loss: 0.0178 - mae: 0.0963 - val_loss: 0.0060 - val_mae: 0.0446
Epoch 4/100
77/77 - 4s - 48ms/step - loss: 0.0150 - mae: 0.0868 - val_loss: 0.0046 - val_mae: 0.0311
Epoch 5/100
77/77 - 5s - 67ms/step - loss: 0.0128 - mae: 0.0776 - val_loss: 0.0041 - val_mae: 0.0321
Epoch 6/100
77/77 - 5s - 69ms/step - loss: 0.0113 - mae: 0.0703 - val_loss: 0.0038 - val_mae: 0.0308
Epoch 7/100
77/77 - 5s - 59ms/step - loss: 0.0105 - mae: 0.0658 - val_loss: 0.0037 - val_mae: 0.0346
Epoch 8/100
77/77 - 3s - 41ms/step - loss: 0.0101 - mae: 0.0632 - val_loss: 0.0034 - val_mae: 0.0317
Epoch 9/100
77/77 - 4s - 47ms/step - loss: 0.0098 - mae: 0.0608 - val_loss: 0.0033 - val_mae: 0.0291
Epoch 10/100
77/77 - 5s - 63ms/step - loss: 0.0096 - mae: 0.0591 - val_loss: 0.0032 - val

In [20]:
lstm_pred = model.predict(X_test, verbose=0).ravel()
evaluate(y_test.ravel(), lstm_pred, "LSTM")

LSTM                   MAE: 0.781   RMSE: 1.749


(0.7811828851699829, np.float64(1.749115720494336))

In [21]:
model.save("../model/temp_lstm.keras")